# Bounded Forex RL pipeline on real EUR/USD data

This notebook fetches a short daily EUR/USD history from Yahoo Finance through the project's `download_eurusd` function, then retrains a deliberately small DQN through `run_walkforward` on two advancing folds. Its saved outputs are empirical results from that bounded real-data run. They are not evidence of a profitable or deployable strategy. Run all cells after installing `.[research]`.


In [1]:
from dataclasses import replace
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
from time import perf_counter
from pathlib import Path
import pandas as pd
import yfinance as yf
from stable_baselines3 import DQN
from forex_rl import ExperimentConfig, download_eurusd, compute_indicators, build_walkforward_folds, make_scaled_env, backtest_model, run_baselines, run_walkforward, generate_walkforward_report, seed_everything
from forex_rl.walk_forward import slice_fold

started = perf_counter()
config = replace(ExperimentConfig(), symbol='EURUSD=X', start_date='2023-01-01',
                 end_date='2023-08-01', random_seed=7, window_size=4, min_train_months=3,
                 val_months=1, test_months=1, step_months=1, n_envs=1, wf_n_envs=1,
                 total_timesteps=96, eval_freq=0, max_bars_in_trade=3,
                 transaction_cost=0.0001, slippage=0.00005)
seed_everything(config.random_seed)
feature_cols = ['ema20', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower', 'atr_n', 'ret']
price_cols = ['open', 'high', 'low', 'close', 'atr']
print('Source: Yahoo Finance via yfinance', yf.__version__, 'ticker:', config.symbol)
print('Requested dates:', config.start_date, 'through', config.end_date, '(end exclusive)')
print('Training steps per fold:', config.total_timesteps, 'DQN learning starts: 8; CPU')
print('Exact experiment config:', config.to_dict())


Source: Yahoo Finance via yfinance 1.7.0 ticker: EURUSD=X
Requested dates: 2023-01-01 through 2023-08-01 (end exclusive)
Training steps per fold: 96 DQN learning starts: 8; CPU
Exact experiment config: {'symbol': 'EURUSD=X', 'start_date': '2023-01-01', 'end_date': '2023-08-01', 'window_size': 4, 'position_risk_frac': 0.01, 'sl_atr_mult': 1.8, 'tp_atr_mult': 2.2, 'max_bars_in_trade': 3, 'transaction_cost': 0.0001, 'slippage': 5e-05, 'invalid_action_penalty': 0.0, 'dd_penalty': 0.0, 'turnover_penalty': 0.0, 'min_train_months': 3, 'val_months': 1, 'test_months': 1, 'step_months': 1, 'use_rolling': True, 'rolling_train_months': 36, 'random_seed': 7, 'algo': 'DQN', 'total_timesteps': 96, 'eval_freq': 0, 'n_envs': 1, 'wf_n_envs': 1, 'rppo_n_steps': 128, 'qrdqn_n_quantiles': 51, 'min_trades_val': 4, 'log_dir': './logs_rl', 'report_dir': './logs_rl/reports'}


In [2]:
retrieved_at = datetime.now(timezone.utc).isoformat()
raw = download_eurusd(config.symbol, config.start_date, config.end_date)
assert 100 <= len(raw) <= 180, 'Expected a bounded real-data sample; inspect source/date range.'
data = compute_indicators(raw)
folds = build_walkforward_folds(data, config.min_train_months, config.val_months,
                                config.test_months, config.step_months)[:2]
assert len(folds) == 2, 'Expected two complete advancing folds.'
print('Retrieved at UTC:', retrieved_at)
print('Source row count and observed dates:', len(raw), raw.index.min(), raw.index.max())
print('Prepared rows:', len(data))
for fold in folds:
    train, validation, test = (slice_fold(data, fold, part) for part in ('train', 'val', 'test'))
    assert train.index.max() < validation.index.min() < test.index.min()
    print('Fold', fold['fold'], 'train/validation/test rows:', len(train), len(validation), len(test))
    print('Fold', fold['fold'], 'boundaries (end exclusive):', fold)


Retrieved at UTC: 2026-09-29T14:00:33.963523+00:00
Source row count and observed dates: 151 2023-01-02 00:00:00 2023-07-31 00:00:00
Prepared rows: 132
Fold 1 train/validation/test rows: 64 22 21
Fold 1 boundaries (end exclusive): {'fold': 1, 'train_start': Timestamp('2023-01-27 00:00:00'), 'train_end': Timestamp('2023-04-27 00:00:00'), 'val_start': Timestamp('2023-04-27 00:00:00'), 'val_end': Timestamp('2023-05-27 00:00:00'), 'test_start': Timestamp('2023-05-27 00:00:00'), 'test_end': Timestamp('2023-06-27 00:00:00')}
Fold 2 train/validation/test rows: 86 21 22
Fold 2 boundaries (end exclusive): {'fold': 2, 'train_start': Timestamp('2023-01-27 00:00:00'), 'train_end': Timestamp('2023-05-27 00:00:00'), 'val_start': Timestamp('2023-05-27 00:00:00'), 'val_end': Timestamp('2023-06-27 00:00:00'), 'test_start': Timestamp('2023-06-27 00:00:00'), 'test_end': Timestamp('2023-07-27 00:00:00')}


In [3]:
trained = []
def train_bounded_fold(algo, train_df, validation_df, fold_config, features, prices, n_envs):
    assert algo == 'DQN' and n_envs == 1
    train_env, validation_env, scaler = make_scaled_env(
        train_df, validation_df, features, prices, fold_config.environment_kwargs,
        fold_config.random_seed, n_envs=n_envs)
    assert scaler.means.equals(train_df[features].mean())
    model = DQN('MlpPolicy', train_env, seed=fold_config.random_seed, device='cpu', verbose=0,
                policy_kwargs={'net_arch': [32, 32]}, learning_rate=0.001,
                buffer_size=256, learning_starts=8, batch_size=8, train_freq=1,
                gradient_steps=1)
    try:
        model.learn(total_timesteps=fold_config.total_timesteps)
    finally:
        train_env.close()
        validation_env.close()
    fold_number = len(trained) + 1
    validation_result = backtest_model(model, scaler, validation_df, features, prices, fold_config)
    trained.append({'fold': fold_number, 'model': model, 'scaler': scaler,
                    'validation': validation_result['metrics'], 'timesteps': model.num_timesteps,
                    'updates': model._n_updates})
    return model, scaler, None

fold_results = run_walkforward(config.algo, data, folds, config, train_bounded_fold, feature_cols, price_cols)
assert len(fold_results) == len(trained) == 2
assert all(item['updates'] > 0 for item in trained)
print('Actual model timesteps and gradient updates per fold:',
      [(item['fold'], item['timesteps'], item['updates']) for item in trained])
print('Walk-forward held-out results:')
print(fold_results[['fold', 'trades', 'total_R', 'sharpe', 'sortino', 'maxDD_pct', 'flat_R', 'rand_R', 'trend_R']].to_string(index=False))


Actual model timesteps and gradient updates per fold: [(1, 96, 88), (2, 96, 88)]
Walk-forward held-out results:
 fold  trades  total_R   sharpe   sortino  maxDD_pct  flat_R  rand_R   trend_R
    1       5 1.073788 2.744719  5.768991  -0.004902     0.0     0.0  1.182249
    2       5 2.162647 7.463889 26.065637  -0.004054     0.0     0.0 -1.925754


In [4]:
validation_summary = pd.DataFrame([
    {'fold': item['fold'], **{key: item['validation'][key]
                             for key in ('trade_count', 'total_R', 'sharpe', 'sortino', 'maxDD_pct')}}
    for item in trained])
print('Per-fold validation results (no model selection):')
print(validation_summary.to_string(index=False))
for fold, item in zip(folds, trained):
    test = slice_fold(data, fold, 'test')
    agent = backtest_model(item['model'], item['scaler'], test, feature_cols, price_cols, config)
    baselines = run_baselines(test, config)
    rows = {'DQN': agent, **baselines}
    summary = pd.DataFrame({name: {'trades': result['metrics']['trade_count'],
                                   'total_R': result['metrics']['total_R'],
                                   'sharpe': result['metrics']['sharpe'],
                                   'sortino': result['metrics']['sortino'],
                                   'maxDD_pct': result['metrics']['maxDD_pct'],
                                   'turnover': result['metrics']['turnover']}
                            for name, result in rows.items()}).T
    print('Fold', fold['fold'], 'held-out test and matched baselines:')
    print(summary.to_string())
with TemporaryDirectory(prefix='forex_rl_bounded_') as report_dir:
    report = generate_walkforward_report(fold_results, report_dir, config.algo)
    assert all(Path(report[key]).is_file() for key in ('csv', 'json', 'html'))
    print('Walk-forward report files generated:', [Path(report[key]).name for key in ('csv', 'json', 'html')])
    print('Aggregate mean:', report['aggregates']['mean'])
print('Elapsed seconds:', round(perf_counter() - started, 2))


Per-fold validation results (no model selection):
 fold  trade_count   total_R    sharpe   sortino  maxDD_pct
    1            2 -0.536297 -3.076819 -3.657169  -0.005356
    2            4  0.505512  1.137180  1.712822  -0.011990
Fold 1 held-out test and matched baselines:
        trades   total_R    sharpe   sortino  maxDD_pct  turnover
DQN        5.0  1.073788  2.744719  5.768991  -0.004902  7.706522
flat       0.0  0.000000       NaN       NaN   0.000000  0.000000
random     0.0  0.000000       NaN       NaN   0.000000  0.000000
trend      5.0  1.182249  3.032024  6.015724  -0.005641  7.725731
Fold 2 held-out test and matched baselines:
        trades   total_R    sharpe    sortino  maxDD_pct  turnover
DQN        5.0  2.162647  7.463889  26.065637  -0.004054  7.517129
flat       0.0  0.000000       NaN        NaN   0.000000  0.000000
random     0.0  0.000000       NaN        NaN   0.000000  0.000000
trend      5.0 -1.925754 -6.439587  -6.900328  -0.019467  7.389798
Walk-forward repo

## Interpretation

These outputs use a real but short EUR/USD daily sample, two advancing walk-forward folds, and 96 DQN training steps per fold. Each fold refits preprocessing on its training partition and retrains DQN before held-out evaluation. The run checks the data, training, evaluation, and aggregate report path. It does not establish statistical significance, production execution realism, stability across market regimes, or profitability. Only the configured DQN path was exercised; QR-DQN, RecurrentPPO, full historical walk-forward, large-model training, model-selection search, and transaction-cost calibration were not performed. Sharpe/Sortino can be undefined on small samples with little return variation.
